UPI SPENDING ANALYSER

1. IMPORT LIBRARIES

In [1]:
import pdfplumber
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

2. DATA EXTRACTION

In [2]:
pdf_path="/home/paragchengappa/programming/UPI-Spending-Analyser/data/raw/bankstats.pdf"

text=""

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        page_text=page.extract_text()
        if page_text:
            text+=page_text+"\n"

print(text[:3000])

Transaction statement
9019517497, paragchengappa@gmail.com
Transactionstatementperiod Sent Received
01January2026-30June2026 ₹77,368.99 ₹70,536
Date&time Transactiondetails Amount
03Jan,2026 PaidtoJAYARAMAPB ₹30
07:41AM UPITransactionID:600358945338
PaidbyStateBankofIndia7202
03Jan,2026 PaidtoSHIVANNA SHIVANNA ₹231
02:17PM UPITransactionID:636962750553
PaidbyStateBankofIndia7202
05Jan,2026 PaidtoPARASHURAMA ₹56.15
07:29AM UPITransactionID:600553332781
PaidbyStateBankofIndia7202
05Jan,2026 PaidtoRAMESHMR ₹5
09:45AM UPITransactionID:600558105853
PaidbyStateBankofIndia7202
05Jan,2026 ReceivedfromMC NALINAKSHI ₹1,100
01:14PM UPITransactionID:969952655934
PaidtoStateBankofIndia7202
05Jan,2026 ReceivedfromMC NALINAKSHI ₹300
01:19PM UPITransactionID:527376183990
PaidtoStateBankofIndia7202
05Jan,2026 PaidtoBlinkit ₹1,112
01:19PM UPITransactionID:637121529011
PaidbyStateBankofIndia7202
06Jan,2026 PaidtoBMTCBUSKA57F1114 ₹15
12:03PM UPITransactionID:600608819250
PaidbyStateBankofIndia7202
Note:Th

In [3]:
lines=text.split("\n")

for line in lines[:40]:
    print(line)

Transaction statement
9019517497, paragchengappa@gmail.com
Transactionstatementperiod Sent Received
01January2026-30June2026 ₹77,368.99 ₹70,536
Date&time Transactiondetails Amount
03Jan,2026 PaidtoJAYARAMAPB ₹30
07:41AM UPITransactionID:600358945338
PaidbyStateBankofIndia7202
03Jan,2026 PaidtoSHIVANNA SHIVANNA ₹231
02:17PM UPITransactionID:636962750553
PaidbyStateBankofIndia7202
05Jan,2026 PaidtoPARASHURAMA ₹56.15
07:29AM UPITransactionID:600553332781
PaidbyStateBankofIndia7202
05Jan,2026 PaidtoRAMESHMR ₹5
09:45AM UPITransactionID:600558105853
PaidbyStateBankofIndia7202
05Jan,2026 ReceivedfromMC NALINAKSHI ₹1,100
01:14PM UPITransactionID:969952655934
PaidtoStateBankofIndia7202
05Jan,2026 ReceivedfromMC NALINAKSHI ₹300
01:19PM UPITransactionID:527376183990
PaidtoStateBankofIndia7202
05Jan,2026 PaidtoBlinkit ₹1,112
01:19PM UPITransactionID:637121529011
PaidbyStateBankofIndia7202
06Jan,2026 PaidtoBMTCBUSKA57F1114 ₹15
12:03PM UPITransactionID:600608819250
PaidbyStateBankofIndia7202
Note:Th

In [12]:
transactions=[]

In [13]:
import re
i=0

while i<len(lines):
    line=lines[i]

    if re.match(r"\d{2}[A-Za-z]{3},\d{4}",line):

        line1=lines[i]
        line2=lines[i+1]
        line3=lines[i+2]

        #Date
        date=re.search(r"\d{2}[A-Za-z]{3},\d{4}",line1).group()

        #Amount
        amount=re.search(r"₹[\d,\.]+",line1).group()

        #Details
        details=line1.replace(date,"")
        details=details.replace(amount,"").strip()

        #Time
        time=re.search(r"\d{2}:\d{2}[AP]M",line2).group()

        #UPI
        upi=re.search(r"\d{12}",line2).group()
        bank=line3

        transactions.append([date,time,details,amount,upi,bank])

        i+=3

    else:
        i+=1


In [14]:
df=pd.DataFrame(transactions,columns=["Date","Time","Details","Amount","UPI_ID","Bank"])
df.head(10)

,Date,Time,Details,Amount,UPI_ID,Bank
0,"03Jan,2026",07:41AM,PaidtoJAYARAMAPB,₹30,600358945338,PaidbyStateBankofIndia7202
1,"03Jan,2026",02:17PM,PaidtoSHIVANNA SHIVANNA,₹231,636962750553,PaidbyStateBankofIndia7202
2,"05Jan,2026",07:29AM,PaidtoPARASHURAMA,₹56.15,600553332781,PaidbyStateBankofIndia7202
3,"05Jan,2026",09:45AM,PaidtoRAMESHMR,₹5,600558105853,PaidbyStateBankofIndia7202
4,"05Jan,2026",01:14PM,ReceivedfromMC NALINAKSHI,"₹1,100",969952655934,PaidtoStateBankofIndia7202
5,"05Jan,2026",01:19PM,ReceivedfromMC NALINAKSHI,₹300,527376183990,PaidtoStateBankofIndia7202
6,"05Jan,2026",01:19PM,PaidtoBlinkit,"₹1,112",637121529011,PaidbyStateBankofIndia7202
7,"06Jan,2026",12:03PM,PaidtoBMTCBUSKA57F1114,₹15,600608819250,PaidbyStateBankofIndia7202
8,"08Jan,2026",11:31AM,PaidtoSmartBazaar BengaluruFR47,₹22.50,600811739250,PaidbyStateBankofIndia7202
9,"08Jan,2026",11:38AM,PaidtoKA01AR2969,₹15,600812237069,PaidbyStateBankofIndia7202


3. DATA CLEANING

In [16]:
df["Date"]=pd.to_datetime(df["Date"],format="%d%b,%Y")

In [17]:
df.dtypes

Date       datetime64[ns]
Time               object
Details            object
Amount             object
UPI_ID             object
Bank               object
dtype: object

In [18]:
df["Amount"]=(df["Amount"]
              .str.replace("₹","",regex=False)
              .str.replace(",","",regex=False)
              .astype(float)
)

4. FEATURE ENGINEERING

In [19]:
df["Transaction_Type"]=df["Details"].apply(
    lambda x:"Paid" if x.startswith("Paid") else "Received"
)

In [20]:
df.head()

,Date,Time,Details,Amount,UPI_ID,Bank,Transaction_Type
0,2026-01-03,07:41AM,PaidtoJAYARAMAPB,30.00,600358945338,PaidbyStateBankofIndia7202,Paid
1,2026-01-03,02:17PM,PaidtoSHIVANNA SHIVANNA,231.00,636962750553,PaidbyStateBankofIndia7202,Paid
2,2026-01-05,07:29AM,PaidtoPARASHURAMA,56.15,600553332781,PaidbyStateBankofIndia7202,Paid
3,2026-01-05,09:45AM,PaidtoRAMESHMR,5.00,600558105853,PaidbyStateBankofIndia7202,Paid
4,2026-01-05,01:14PM,ReceivedfromMC NALINAKSHI,1100.00,969952655934,PaidtoStateBankofIndia7202,Received


In [21]:
df["Merchant"]=(df["Details"]
                .str.replace("Paidto","",regex=False)
                .str.replace("Receivedfrom","",regex=False)
                .str.strip()
                )

In [22]:
df.head()

,Date,Time,Details,Amount,UPI_ID,Bank,Transaction_Type,Merchant
0,2026-01-03,07:41AM,PaidtoJAYARAMAPB,30.00,600358945338,PaidbyStateBankofIndia7202,Paid,JAYARAMAPB
1,2026-01-03,02:17PM,PaidtoSHIVANNA SHIVANNA,231.00,636962750553,PaidbyStateBankofIndia7202,Paid,SHIVANNA SHIVANNA
2,2026-01-05,07:29AM,PaidtoPARASHURAMA,56.15,600553332781,PaidbyStateBankofIndia7202,Paid,PARASHURAMA
3,2026-01-05,09:45AM,PaidtoRAMESHMR,5.00,600558105853,PaidbyStateBankofIndia7202,Paid,RAMESHMR
4,2026-01-05,01:14PM,ReceivedfromMC NALINAKSHI,1100.00,969952655934,PaidtoStateBankofIndia7202,Received,MC NALINAKSHI


In [23]:
df["Month"]=df["Date"].dt.month_name()

In [24]:
df["Day"]=df["Date"].dt.day_name()

5. EXPLORATORY DATA ANALYSIS

#MONTHLY SPENDING

#CATEGORY- WISE SPENDING

#TOP MERCHANTS

#SPENDING DISTRIBUTION

#SPENDING BY WEEKDAY

#LARGEST TRANSACTIONS

6. KEY INSIGHTS

7. LIMITATIONS